# 10장 · 계수는 효과가 아니다: 심슨의 역설

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [10장 · 계수는 효과가 아니다: 심슨의 역설](https://grow.minds.kr/textbooks/css-methods/causal/book/ch10-계수는-효과가-아니다.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 심슨의 역설
저그마을을 직접 짓습니다: 수컷은 보양제를 더 먹고 기력도 원래 높다(성별 = 공통 원인). 전체로 보면 양(+), 성별로 나누면 음(-).

In [ ]:
def make_simpson(seed=73, n=400):
    rng = np.random.default_rng(seed)
    sex = rng.integers(0, 2, n)                          # 0=암컷, 1=수컷
    dose = np.clip(rng.normal(2 + 4*sex, 1.5), 0, 10)    # 수컷이 더 복용
    vigor = 55 + 20*sex - 1.5*dose + rng.normal(0, 4, n) # 진짜 효과 = 해로움(-1.5)
    return pd.DataFrame({"sex": sex, "dose": dose.round(1), "vigor": vigor.round(1)})
zg = make_simpson()                       # 씨앗 73, 400마리
print("전체 기울기:", round(np.polyfit(zg.dose, zg.vigor,1)[0], 2))          # +1.70
for sx,name in [(0,"암컷"),(1,"수컷")]:
    g = zg[zg.sex==sx]
    print(f"  {name} 기울기:", round(np.polyfit(g.dose, g.vigor,1)[0], 2))    # -1.72 / -1.66

## 3. 극단값 시험
성별의 기력 차이(+20)를 0으로 하면 전체 기울기가 진짜 값(-1.5쪽)으로 돌아옵니다.

In [ ]:
zg2 = make_simpson(); import numpy as _np
# 성별 효과 제거: 수컷 기력에서 +20을 빼 균질화
zg2.loc[zg2.sex==1, "vigor"] = zg2.loc[zg2.sex==1, "vigor"] - 20
print("성별 효과 제거 후 전체 기울기:", round(np.polyfit(zg2.dose, zg2.vigor,1)[0], 2))

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 *검증*이 이 책의 핵심입니다.